# Ficha AP — Aula 3.4 — CNN, Filtros e Feature Maps — Resolvida

**Tema:** Visualização de filtros e de *feature maps* na classificação de imagens com CNNs usando MNIST.

Este notebook resolve as tarefas T1 a T10:
- download e preparação do MNIST;
- normalização e *holdout*;
- visualização de dados;
- definição de 4 modelos CNN;
- treino com `CrossEntropyLoss` e `SGD`;
- avaliação com previsões e matriz de confusão;
- uso dos modelos para previsão, visualização de filtros, parâmetros, *max pooling* e *feature maps*.

> Nota: se tiveres os ficheiros `CNNModel_1.pth`, `CNNModel_2.pth`, `CNNModel_3.pth` e `CNNModel_4.pth` da Blackboard, coloca-os na mesma pasta deste notebook. O notebook também consegue treinar os modelos do zero e gravar esses `.pth`.


## 0. Setup

Executa esta célula primeiro. Em Google Colab, muda o runtime para GPU:  
`Runtime > Change runtime type > GPU`.


In [ ]:
import sys
import subprocess
import importlib

# Instala apenas o que estiver em falta.
# Em Colab, torch/torchvision normalmente já vêm instalados.
required_packages = {
    "torch": "torch",
    "torchvision": "torchvision",
    "torchinfo": "torchinfo",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "gdown": "gdown",
}

for module_name, package_name in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        print(f"A instalar {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

print("Setup concluído.")


In [ ]:
import os
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
from torchinfo import summary

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Constantes pedidas no enunciado
BATCH_SIZE = 32
LEARNING_RATE = 0.001

# O enunciado pede: CNN1/2/3 com 15 epochs; CNN4 com 75 epochs.
EPOCHS = {
    "CNNModel_1": 15,
    "CNNModel_2": 15,
    "CNNModel_3": 15,
    "CNNModel_4": 75,
}

# Deixa False para fazer a ficha completa.
# Mete True só para testar rapidamente o notebook.
FAST_DEV_RUN = False
FAST_TRAIN_SIZE = 2000
FAST_VAL_SIZE = 500
FAST_TEST_SIZE = 500

# Se True, treina sempre do zero, mesmo que existam .pth.
# Se False, tenta carregar .pth existentes e só treina se não existirem.
FORCE_TRAIN = True

DATA_DIR = Path("./data")
MODEL_DIR = Path("./models")
MODEL_DIR.mkdir(exist_ok=True)

# Se o professor der links diretos/Google Drive para os modelos da BB, coloca-os aqui.
# Se ficarem vazios, o notebook treina os modelos do zero.
MODEL_URLS = {
    "CNNModel_1.pth": "",
    "CNNModel_2.pth": "",
    "CNNModel_3.pth": "",
    "CNNModel_4.pth": "",
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 1. Preparar os dados — T1 a T4

Tarefas cobertas aqui:
- T1: download do MNIST;
- T3: `batch_size = 32`;
- T4.1: funções de transformação;
- T4.2: normalização;
- T4.3: `DataLoader` com *holdout*.


In [ ]:
# T4.1 e T4.2 — transformação + normalização.
# Valores clássicos de média/desvio-padrão do MNIST.
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

# T1 — download do dataset MNIST.
train_dataset_full = datasets.MNIST(
    root=str(DATA_DIR),
    train=True,
    download=True,
    transform=train_transform,
)

test_dataset = datasets.MNIST(
    root=str(DATA_DIR),
    train=False,
    download=True,
    transform=test_transform,
)

print("Total treino original:", len(train_dataset_full))
print("Total teste:", len(test_dataset))


In [ ]:
# T4.3 — holdout: 80% treino / 20% validação a partir das 60 000 imagens de treino.
train_size = int(0.8 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

train_dataset, val_dataset = random_split(
    train_dataset_full,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)

if FAST_DEV_RUN:
    train_dataset = Subset(train_dataset, range(min(FAST_TRAIN_SIZE, len(train_dataset))))
    val_dataset = Subset(val_dataset, range(min(FAST_VAL_SIZE, len(val_dataset))))
    test_dataset = Subset(test_dataset, range(min(FAST_TEST_SIZE, len(test_dataset))))

train_dl = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
val_dl = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())
test_dl = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

print("Treino:", len(train_dataset))
print("Validação:", len(val_dataset))
print("Teste:", len(test_dataset))
print("Batch size:", BATCH_SIZE)


## 1.1 Visualizar os dados — T5

Aqui colocamos o label por extenso e visualizamos um batch de treino.


In [ ]:
LABEL_NAMES = {
    0: "zero",
    1: "um",
    2: "dois",
    3: "três",
    4: "quatro",
    5: "cinco",
    6: "seis",
    7: "sete",
    8: "oito",
    9: "nove",
}

def output_label(label):
    """Recebe um inteiro/tensor e devolve o label por extenso."""
    if torch.is_tensor(label):
        label = int(label.item())
    return LABEL_NAMES[int(label)]

def denormalize(img):
    """Inverte a normalização para mostrar imagens de forma mais natural."""
    mean = torch.tensor([0.1307]).view(1, 1, 1)
    std = torch.tensor([0.3081]).view(1, 1, 1)
    return img.cpu() * std + mean

def visualize_mnist_batch(dataloader, num_images=16):
    images, labels = next(iter(dataloader))
    num_images = min(num_images, len(images))
    cols = 4
    rows = int(np.ceil(num_images / cols))
    plt.figure(figsize=(10, 2.5 * rows))
    for i in range(num_images):
        plt.subplot(rows, cols, i + 1)
        img = denormalize(images[i]).squeeze()
        plt.imshow(img, cmap="gray")
        plt.title(f"{int(labels[i])} — {output_label(labels[i])}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

visualize_mnist_batch(train_dl, num_images=16)


## 2. Definir os modelos — T6

São definidos os 4 modelos pedidos no enunciado.


In [ ]:
class CNNModel_1(nn.Module):
    """
    CNN 1:
    - Sequential composed layer: Conv2d + ReLU + MaxPool2d
    - Sequential composed layer: Conv2d + ReLU + MaxPool2d
    - Linear + ReLU + Linear + Softmax
    """
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.fc1 = nn.Linear(32 * 5 * 5, 100)
        self.act1 = nn.ReLU()
        self.fc2 = nn.Linear(100, 10)
        self.act2 = nn.Softmax(dim=1)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.view(out.size(0), -1)
        out = self.fc1(out)
        out = self.act1(out)
        out = self.fc2(out)
        out = self.act2(out)
        return out


class CNNModel_2(nn.Module):
    """
    CNN 2:
    - Sequential composed layer: Conv2d + ReLU + MaxPool2d
    - Sequential composed layer: Conv2d + ReLU + MaxPool2d
    - Linear
    """
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.fc1 = nn.Linear(32 * 5 * 5, 10)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.view(out.size(0), -1)
        out = self.fc1(out)
        return out


class CNNModel_3(nn.Module):
    """
    CNN 3:
    - Sequential composed layer: Conv2d + BatchNorm2d + ReLU + MaxPool2d
    - Sequential composed layer: Conv2d + BatchNorm2d + ReLU + MaxPool2d
    - Linear + Dropout + Linear + Linear
    """
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.fc1 = nn.Linear(64 * 5 * 5, 625)
        self.dropout = nn.Dropout(p=0.25)
        self.fc2 = nn.Linear(625, 128)
        self.fc3 = nn.Linear(128, 10)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.view(out.size(0), -1)
        out = F.relu(self.fc1(out))
        out = self.dropout(out)
        out = F.relu(self.fc2(out))
        out = self.fc3(out)
        return out


class CNNModel_4(nn.Module):
    """
    CNN 4:
    - Sequential composed layer: Conv2d + BatchNorm2d + ReLU + MaxPool2d + Dropout
    - Sequential composed layer: Conv2d + BatchNorm2d + ReLU + MaxPool2d + Dropout
    - Linear + Linear
    """
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(p=0.10),
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(p=0.10),
        )
        self.fc1 = nn.Linear(64 * 5 * 5, 100)
        self.fc2 = nn.Linear(100, 10)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.view(out.size(0), -1)
        out = F.relu(self.fc1(out))
        out = self.fc2(out)
        return out


In [ ]:
# Resumo dos modelos
for model_cls in [CNNModel_1, CNNModel_2, CNNModel_3, CNNModel_4]:
    model = model_cls().to(device)
    print("\n" + "=" * 80)
    print(model_cls.__name__)
    print(summary(model, input_size=(BATCH_SIZE, 1, 28, 28), verbose=0))


## 3. Treinar os modelos — T7

Configuração pedida:
- `CrossEntropyLoss`;
- `SGD`;
- `learning_rate = 0.001`;
- CNN 1, 2 e 3: 15 epochs;
- CNN 4: 75 epochs.


In [ ]:
def accuracy_from_outputs(outputs, labels):
    preds = outputs.argmax(dim=1)
    return (preds == labels).float().mean().item()

@torch.no_grad()
def evaluate_loss_accuracy(model, dataloader, criterion):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    for xb, yb in dataloader:
        xb = xb.to(device)
        yb = yb.to(device)
        outputs = model(xb)
        loss = criterion(outputs, yb)
        batch_size = xb.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (outputs.argmax(dim=1) == yb).sum().item()
        total_examples += batch_size
    return total_loss / total_examples, total_correct / total_examples

def train_one_epoch(model, dataloader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    for xb, yb in dataloader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        batch_size = xb.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (outputs.argmax(dim=1) == yb).sum().item()
        total_examples += batch_size

    return total_loss / total_examples, total_correct / total_examples

def fit_model(model, train_dl, val_dl, epochs, lr, model_name):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    best_val_acc = -1.0
    best_state = copy.deepcopy(model.state_dict())
    history = []

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_dl, criterion, optimizer)
        val_loss, val_acc = evaluate_loss_accuracy(model, val_dl, criterion)

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
        }
        history.append(row)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"{model_name} | epoch {epoch:03d}/{epochs:03d} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
        )

    model.load_state_dict(best_state)
    save_path = MODEL_DIR / f"{model_name}.pth"
    torch.save(model.state_dict(), save_path)
    print(f"Melhor modelo guardado em: {save_path}")

    return pd.DataFrame(history)

def plot_history(history_df, model_name):
    plt.figure(figsize=(8, 4))
    plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
    plt.title(f"Loss — {model_name}")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.plot(history_df["epoch"], history_df["train_accuracy"], label="train_accuracy")
    plt.plot(history_df["epoch"], history_df["val_accuracy"], label="val_accuracy")
    plt.title(f"Accuracy — {model_name}")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
def try_download_model(filename):
    """Tenta descarregar o modelo se for dado um URL em MODEL_URLS."""
    local_path = MODEL_DIR / filename
    if local_path.exists():
        return local_path

    url = MODEL_URLS.get(filename, "").strip()
    if not url:
        return None

    import gdown
    print(f"A descarregar {filename}...")
    try:
        gdown.download(url, str(local_path), quiet=False, fuzzy=True)
        return local_path if local_path.exists() else None
    except Exception as exc:
        print(f"Não foi possível descarregar {filename}: {exc}")
        return None


def load_model_from_pth(model_cls, model_name):
    """Carrega um modelo .pth se existir. Suporta state_dict ou modelo completo."""
    filename = f"{model_name}.pth"
    path = MODEL_DIR / filename

    if not path.exists():
        # Também aceita .pth na pasta atual.
        alt_path = Path(filename)
        if alt_path.exists():
            path = alt_path
        else:
            downloaded = try_download_model(filename)
            if downloaded is None:
                return None
            path = downloaded

    print(f"A carregar {path}...")
    model = model_cls().to(device)

    try:
        try:
            obj = torch.load(path, map_location=device, weights_only=False)
        except TypeError:
            obj = torch.load(path, map_location=device)

        if isinstance(obj, nn.Module):
            model = obj.to(device)
        elif isinstance(obj, dict):
            state_dict = obj.get("model_state_dict", obj)
            try:
                model.load_state_dict(state_dict)
            except RuntimeError:
                model.load_state_dict(state_dict, strict=False)
        else:
            print("Formato .pth não reconhecido. O modelo será treinado do zero.")
            return None

        model.eval()
        return model
    except Exception as exc:
        print(f"Erro ao carregar {path}: {exc}")
        return None


In [ ]:
MODEL_CLASSES = {
    "CNNModel_1": CNNModel_1,
    "CNNModel_2": CNNModel_2,
    "CNNModel_3": CNNModel_3,
    "CNNModel_4": CNNModel_4,
}

trained_models = {}
histories = {}

for model_name, model_cls in MODEL_CLASSES.items():
    print("\n" + "#" * 90)
    print(f"### {model_name}")

    model = None
    if not FORCE_TRAIN:
        model = load_model_from_pth(model_cls, model_name)

    if model is None:
        print(f"A treinar {model_name} do zero...")
        model = model_cls().to(device)
        epochs = EPOCHS[model_name]
        history_df = fit_model(
            model=model,
            train_dl=train_dl,
            val_dl=val_dl,
            epochs=epochs,
            lr=LEARNING_RATE,
            model_name=model_name,
        )
        histories[model_name] = history_df
        display(history_df.tail())
        plot_history(history_df, model_name)
    else:
        print(f"{model_name} carregado de .pth.")
        histories[model_name] = None

    trained_models[model_name] = model.to(device).eval()

print("Modelos disponíveis:", list(trained_models.keys()))


## 4. Avaliar os modelos — T8

Nesta secção avaliamos cada modelo no conjunto de teste, mostrando:
- accuracy;
- relatório de classificação;
- matriz de confusão;
- imagens com previsões.


In [ ]:
@torch.no_grad()
def get_predictions(model, dataloader):
    model.eval()
    all_preds = []
    all_labels = []
    all_images = []

    for xb, yb in dataloader:
        xb = xb.to(device)
        outputs = model(xb)
        preds = outputs.argmax(dim=1).cpu()

        all_preds.extend(preds.numpy().tolist())
        all_labels.extend(yb.numpy().tolist())
        all_images.append(xb.cpu())

    all_images = torch.cat(all_images, dim=0)
    return np.array(all_labels), np.array(all_preds), all_images

def display_confusion_matrix(y_true, y_pred, model_name):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=list(LABEL_NAMES.keys()),
        yticklabels=list(LABEL_NAMES.keys()),
    )
    plt.title(f"Matriz de confusão — {model_name}")
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.show()


def display_predictions(images, y_true, y_pred, model_name, num_images=16):
    num_images = min(num_images, len(images))
    cols = 4
    rows = int(np.ceil(num_images / cols))
    plt.figure(figsize=(11, 2.8 * rows))
    for i in range(num_images):
        plt.subplot(rows, cols, i + 1)
        img = denormalize(images[i]).squeeze()
        plt.imshow(img, cmap="gray")
        title = f"Real: {y_true[i]} ({output_label(y_true[i])})\nPred: {y_pred[i]} ({output_label(y_pred[i])})"
        plt.title(title)
        plt.axis("off")
    plt.suptitle(f"Previsões — {model_name}", y=1.02)
    plt.tight_layout()
    plt.show()


In [ ]:
evaluation_rows = []

for model_name, model in trained_models.items():
    print("\n" + "=" * 90)
    print(f"Avaliação: {model_name}")
    y_true, y_pred, test_images = get_predictions(model, test_dl)
    acc = accuracy_score(y_true, y_pred)
    evaluation_rows.append({"model": model_name, "test_accuracy": acc})
    print(f"Accuracy no teste: {acc:.4f}")
    print(classification_report(y_true, y_pred, target_names=[output_label(i) for i in range(10)]))
    display_confusion_matrix(y_true, y_pred, model_name)
    display_predictions(test_images, y_true, y_pred, model_name, num_images=16)

results_df = pd.DataFrame(evaluation_rows).sort_values("test_accuracy", ascending=False)
display(results_df)


## 5. Usar o modelo e visualizar filtros / feature maps — T9

Nesta secção são criadas funções para:
- prever um caso dado;
- apresentar batch de imagens previstas;
- imprimir o modelo;
- visualizar parâmetros por camada;
- visualizar filtros das camadas convolucionais;
- aplicar max pooling;
- obter ativações das camadas convolucionais e de pooling;
- visualizar feature maps parciais.


In [ ]:
def img_show(img, legenda):
    """Mostra uma imagem MNIST."""
    if img.ndim == 4:
        img = img[0]
    img = denormalize(img.detach().cpu()).squeeze()
    plt.figure(figsize=(3, 3))
    plt.axis("off")
    plt.title(legenda)
    plt.imshow(img, cmap="gray")
    plt.show()

@torch.no_grad()
def make_prediction(model, img):
    """Prevê a classe de uma única imagem."""
    model.eval()
    img = img.reshape(1, 1, 28, 28).to(device)
    outputs = model(img)
    prediction = int(outputs.argmax(dim=1).cpu().item())
    legenda = f"predict: {prediction} — {output_label(prediction)}"
    print("Shape:", img.shape)
    print("Dtype:", img.dtype)
    img_show(img.cpu(), legenda)
    return prediction

# Exemplo: prever a quarta imagem do primeiro batch de teste.
imagens, labels = next(iter(test_dl))
example_pred = make_prediction(trained_models["CNNModel_1"], imagens[3])
print("Predição:", example_pred)
print("Label real:", int(labels[3]), output_label(labels[3]))


In [ ]:
@torch.no_grad()
def show_batch_images_predictions(model, dataloader, num_images=16):
    model.eval()
    images, labels = next(iter(dataloader))
    images_device = images.to(device)
    outputs = model(images_device)
    preds = outputs.argmax(dim=1).cpu()

    num_images = min(num_images, len(images))
    cols = 4
    rows = int(np.ceil(num_images / cols))
    plt.figure(figsize=(11, 2.8 * rows))
    for i in range(num_images):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(denormalize(images[i]).squeeze(), cmap="gray")
        plt.title(f"Real: {int(labels[i])}\nPred: {int(preds[i])}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()
    return images, labels, preds

for model_name, model in trained_models.items():
    print("\n" + "=" * 90)
    print(model_name)
    batch_images, batch_labels, batch_preds = show_batch_images_predictions(model, test_dl, num_images=16)


In [ ]:
def print_model_and_parameters(model, model_name):
    print("=" * 90)
    print(model_name)
    print(model)
    print("\nParâmetros por camada:")

    rows = []
    total_params = 0
    trainable_params = 0
    for name, param in model.named_parameters():
        count = param.numel()
        total_params += count
        if param.requires_grad:
            trainable_params += count
        rows.append({
            "name": name,
            "shape": tuple(param.shape),
            "num_params": count,
            "trainable": param.requires_grad,
        })

    df = pd.DataFrame(rows)
    display(df)
    print("Total params:", total_params)
    print("Trainable params:", trainable_params)
    return df

parameter_tables = {}
for model_name, model in trained_models.items():
    parameter_tables[model_name] = print_model_and_parameters(model, model_name)


### 5.1 Visualizar filtros / pesos das camadas convolucionais


In [ ]:
def plot_filters_single_channel_big(t):
    nrows = t.shape[0] * t.shape[2]
    ncols = t.shape[1] * t.shape[3]
    npimg = np.array(t.detach().cpu().numpy(), np.float32)
    npimg = npimg.transpose((0, 2, 1, 3))
    npimg = npimg.ravel().reshape(nrows, ncols)
    npimg = npimg.T
    fig, ax = plt.subplots(figsize=(max(ncols / 6, 4), max(nrows / 20, 2)))
    sns.heatmap(npimg, xticklabels=False, yticklabels=False, cmap="gray", ax=ax, cbar=False)
    plt.show()


def plot_filters_single_channel(t):
    nplots = t.shape[0] * t.shape[1]
    ncols = 12
    nrows = int(np.ceil(nplots / ncols))
    count = 0
    fig = plt.figure(figsize=(ncols, max(nrows, 1)))

    for i in range(t.shape[0]):
        for j in range(t.shape[1]):
            count += 1
            ax = fig.add_subplot(nrows, ncols, count)
            npimg = np.array(t[i, j].detach().cpu().numpy(), np.float32)
            std = np.std(npimg)
            if std > 0:
                npimg = (npimg - np.mean(npimg)) / std
            npimg = np.minimum(1, np.maximum(0, (npimg + 0.5)))
            ax.imshow(npimg, cmap="gray")
            ax.set_title(f"{i},{j}", fontsize=8)
            ax.axis("off")
    plt.tight_layout()
    plt.show()


def plot_filters_multi_channel(t):
    num_kernels = t.shape[0]
    num_cols = 12
    num_rows = int(np.ceil(num_kernels / num_cols))
    fig = plt.figure(figsize=(num_cols, max(num_rows, 1)))

    for i in range(num_kernels):
        ax = fig.add_subplot(num_rows, num_cols, i + 1)
        npimg = np.array(t[i].detach().cpu().numpy(), np.float32)
        std = np.std(npimg)
        if std > 0:
            npimg = (npimg - np.mean(npimg)) / std
        npimg = np.minimum(1, np.maximum(0, (npimg + 0.5)))
        npimg = npimg.transpose((1, 2, 0))
        ax.imshow(npimg)
        ax.axis("off")
        ax.set_title(str(i), fontsize=8)

    plt.tight_layout()
    plt.savefig("kernels.png", dpi=100)
    plt.show()


def plot_weights(layer, single_channel=True, collated=False):
    if isinstance(layer, nn.Conv2d):
        weight_tensor = layer.weight.data
        if single_channel:
            if collated:
                plot_filters_single_channel_big(weight_tensor)
            else:
                plot_filters_single_channel(weight_tensor)
        else:
            if weight_tensor.shape[1] == 3:
                plot_filters_multi_channel(weight_tensor)
            else:
                print("Só é possível visualizar como multi-canal quando a camada tem 3 canais de entrada.")
    else:
        print("Só é possível visualizar camadas convolucionais.")


def get_conv_layers(model):
    return [(name, module) for name, module in model.named_modules() if isinstance(module, nn.Conv2d)]

# Visualizar os filtros de todas as camadas convolucionais de todos os modelos.
for model_name, model in trained_models.items():
    print("\n" + "=" * 90)
    print(f"Filtros — {model_name}")
    for layer_name, layer in get_conv_layers(model):
        print(f"Camada convolucional: {layer_name} | pesos: {tuple(layer.weight.shape)}")
        plot_weights(layer, single_channel=True)


### 5.2 Aplicar max pooling


In [ ]:
def apply_max_pooling_demo(img):
    """Aplica MaxPool2d a uma imagem e mostra o antes/depois."""
    pool = nn.MaxPool2d(kernel_size=2, stride=2)
    img_4d = img.reshape(1, 1, 28, 28)
    pooled = pool(img_4d)

    plt.figure(figsize=(7, 3))
    plt.subplot(1, 2, 1)
    plt.imshow(denormalize(img_4d[0]).squeeze(), cmap="gray")
    plt.title("Original 28x28")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(pooled[0, 0].detach().cpu(), cmap="gray")
    plt.title("MaxPool 14x14")
    plt.axis("off")

    plt.tight_layout()
    plt.show()
    print("Shape original:", tuple(img_4d.shape))
    print("Shape após MaxPool:", tuple(pooled.shape))
    return pooled

imagens, labels = next(iter(test_dl))
pooled_example = apply_max_pooling_demo(imagens[0])


### 5.3 Percorrer camadas convolucionais/pooling e visualizar feature maps

Usamos *forward hooks* para guardar as ativações reais do modelo durante o `forward`. Assim funciona também em modelos com `BatchNorm`, `ReLU` e `Dropout`.


In [ ]:
def get_conv_pool_layers(model):
    return [(name, module) for name, module in model.named_modules() if isinstance(module, (nn.Conv2d, nn.MaxPool2d))]

@torch.no_grad()
def get_feature_maps(model, images, layer_types=(nn.Conv2d, nn.MaxPool2d)):
    model.eval()
    activations = []
    hooks = []

    def make_hook(layer_name):
        def hook(module, inputs, output):
            activations.append({
                "name": layer_name,
                "type": module.__class__.__name__,
                "activation": output.detach().cpu(),
            })
        return hook

    for name, module in model.named_modules():
        if isinstance(module, layer_types):
            hooks.append(module.register_forward_hook(make_hook(name)))

    _ = model(images.to(device))

    for hook_handle in hooks:
        hook_handle.remove()

    return activations


def print_feature_map_shapes(feature_maps):
    print(f"Obtiveram-se {len(feature_maps)} tensores com o shape:")
    for i, item in enumerate(feature_maps):
        print(f"Layer {i} | {item['name']} ({item['type']}) - {tuple(item['activation'].shape)}")


def visualize_feature_maps_partial(feature_maps, num_imagem=0, max_filters=18):
    for num_layer, item in enumerate(feature_maps):
        layer_viz = item["activation"][num_imagem]
        num_filters = min(max_filters, layer_viz.shape[0])

        cols = 6
        rows = int(np.ceil(num_filters / cols))
        plt.figure(figsize=(14, 2.3 * rows))
        print(f"Layer {num_layer + 1}: {item['name']} ({item['type']})")

        for i in range(num_filters):
            plt.subplot(rows, cols, i + 1)
            plt.imshow(layer_viz[i], cmap="gray")
            plt.title(f"filter {i}", fontsize=8)
            plt.axis("off")

        plt.tight_layout()
        plt.show()
        plt.close()

# Exemplo para todos os modelos, usando um batch do conjunto de teste.
images, labels = next(iter(test_dl))

for model_name, model in trained_models.items():
    print("\n" + "=" * 90)
    print(f"Feature maps — {model_name}")
    conv_pool_layers = get_conv_pool_layers(model)
    print("Camadas Conv/Pool encontradas:")
    for name, layer in conv_pool_layers:
        print(" -", name, layer)

    feature_maps = get_feature_maps(model, images)
    print_feature_map_shapes(feature_maps)
    visualize_feature_maps_partial(feature_maps, num_imagem=0, max_filters=18)


## 6. Conclusão — T10

O notebook apresenta os resultados das tarefas executadas:
- dataset MNIST preparado com normalização e holdout;
- batch de imagens visualizado com labels por extenso;
- quatro modelos CNN definidos;
- treino com accuracy/loss de treino e validação;
- avaliação com previsões e matriz de confusão;
- uso dos modelos com visualização de parâmetros, filtros, max pooling e feature maps.

Depois de correres tudo, guarda/submete este `.ipynb` com os outputs gerados.
